# Vanilla LightFM - User Split (80/20)
This notebook keeps the same preprocessing and vanilla LightFM training pipeline, but uses a train-test split by user ID (80/20) without a validation split.

## 1. Environment Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
# Keep linear algebra backends single-threaded for stable timing and reproducibility.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

from pathlib import Path

import numpy as np
import pandas as pd

# Global seed used for all randomized operations.
np.random.seed(42)

## 2. Load and Preprocess Data

In [2]:
# All baseline methods should consume the same input CSV files for comparability.
DATA_DIR = Path(".")
MOVIES_PATH = DATA_DIR / "movies.csv"
RATINGS_PATH = DATA_DIR / "ratings.csv"
USERS_PATH = DATA_DIR / "user_profiles.csv"

for required_path in [MOVIES_PATH, RATINGS_PATH, USERS_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required file: {required_path}")

movies_df = pd.read_csv(MOVIES_PATH)
ratings_df = pd.read_csv(RATINGS_PATH)
user_profiles_df = pd.read_csv(USERS_PATH)

# Standardize table schemas.
ratings = ratings_df.rename(columns={"id": "user_id", "rate": "rating"}).copy()
movies = movies_df.copy()
users = user_profiles_df.rename(columns={"id": "user_id"}).copy()

ratings = ratings[["user_id", "movie_id", "rating"]].dropna()
movies = movies.drop_duplicates(subset=["id"]).copy()
users = users.drop_duplicates(subset=["user_id"]).copy()

# Keep only user flags available in the current file.
flag_cols = [c for c in ["early_bird", "night_owl", "weekend_tweeter"] if c in users.columns]
if "followers_count" not in users.columns:
    raise ValueError("Expected a followers_count column in user_profiles.csv")

# Ensure optional item metadata fields are always present.
for col in ["genres", "actors", "directors"]:
    if col not in movies.columns:
        movies[col] = ""

movies["genres"] = movies["genres"].fillna("")
movies["actors"] = movies["actors"].fillna("")
movies["directors"] = movies["directors"].fillna("")
movies["year_published"] = pd.to_numeric(movies.get("year_published"), errors="coerce")

def split_tokens(value):
    # Parse pipe-separated metadata, for example: "Action|Drama".
    if pd.isna(value) or value == "":
        return []
    return [t.strip() for t in str(value).split("|") if t.strip()]

print(f"Movies: {movies.shape[0]:,} | Ratings: {ratings.shape[0]:,} | Users: {users.shape[0]:,}")

Movies: 78,628 | Ratings: 1,172,038 | Users: 482


## 3. Split Train-Test by User ID (80/20)

In [3]:
def split_users_train_test(ratings_df, test_size=0.20, seed=42):
    # Split by unique user IDs so each user belongs to exactly one split.
    unique_users = ratings_df["user_id"].dropna().astype(int).unique().copy()
    if len(unique_users) < 2:
        raise ValueError("At least two users are required for an 80/20 user split.")

    rng = np.random.default_rng(seed)
    rng.shuffle(unique_users)

    n_train_users = int(round(len(unique_users) * (1.0 - test_size)))
    n_train_users = min(max(1, n_train_users), len(unique_users) - 1)

    train_users = unique_users[:n_train_users]
    test_users = unique_users[n_train_users:]

    train_df = ratings_df[ratings_df["user_id"].astype(int).isin(train_users)].copy()
    test_df = ratings_df[ratings_df["user_id"].astype(int).isin(test_users)].copy()

    return train_df, test_df, train_users, test_users

def build_user_pos_lookup(df):
    # Build dictionary: user_id -> set(positive movie_ids).
    lookup = {}
    for uid, group in df.groupby("user_id"):
        lookup[int(uid)] = set(group["movie_id"].astype(int).tolist())
    return lookup

def merge_user_pos_lookups(*lookups):
    # Merge multiple user->set(item) maps into one map.
    merged = {}
    for lookup in lookups:
        for uid, items in lookup.items():
            if uid not in merged:
                merged[uid] = set()
            merged[uid].update(items)
    return merged

train_df, test_df, train_users, test_users = split_users_train_test(ratings, test_size=0.20, seed=42)

train_user_pos = build_user_pos_lookup(train_df)
test_user_pos = build_user_pos_lookup(test_df)
all_user_pos = merge_user_pos_lookups(train_user_pos, test_user_pos)

all_item_ids = sorted(movies["id"].dropna().astype(int).unique().tolist())
all_item_set = set(all_item_ids)

print(f"Split users -> train: {len(train_users):,}, test: {len(test_users):,}")
print(f"Split rows  -> train: {len(train_df):,}, test: {len(test_df):,}")

Split users -> train: 489, test: 122
Split rows  -> train: 938,844, test: 233,194


## 4. Ranking Evaluation Utilities

In [4]:
def sample_negatives(candidate_pool, exclude_set, n_negatives=20, rng=None):
    # Sample negative candidate items while excluding known positives.
    rng = rng or np.random.default_rng(42)
    pool = np.array([i for i in candidate_pool if i not in exclude_set], dtype=int)
    if len(pool) == 0:
        return np.array([], dtype=int)
    replace = len(pool) < n_negatives
    return rng.choice(pool, size=n_negatives, replace=replace)

def ranking_metrics_from_rank(rank, k=10):
    # One positive item per evaluation row.
    recall = 1.0 if rank <= k else 0.0
    ndcg = 1.0 / np.log2(rank + 1) if rank <= k else 0.0
    mrr = 1.0 / rank
    return recall, ndcg, mrr

def rank_positive_among_candidates(scores, positive_mask, rng=None):
    # Randomly break ties to avoid deterministic order bias.
    rng = rng or np.random.default_rng(42)
    positive_idx = int(np.flatnonzero(positive_mask)[0])
    pos_score = float(scores[positive_idx])

    better = int(np.sum(scores > pos_score))
    tie_indices = np.flatnonzero(scores == pos_score)
    tie_count = int(len(tie_indices))

    if tie_count <= 1:
        return better + 1, tie_count

    shuffled_ties = tie_indices[rng.permutation(tie_count)]
    tie_pos = int(np.where(shuffled_ties == positive_idx)[0][0])
    return better + tie_pos + 1, tie_count

def evaluate_ranker(score_fn, positives_df, candidate_pool, all_user_pos_lookup, n_negatives=20, k=10, seed=42):
    # Evaluate using one positive plus sampled negatives for each interaction.
    rng = np.random.default_rng(seed)
    rows = []
    tie_cases = 0

    for uid, group in positives_df.groupby("user_id"):
        uid = int(uid)
        known_positives = all_user_pos_lookup.get(uid, set())

        for _, row in group.iterrows():
            pos_item = int(row["movie_id"])
            exclude = set(known_positives)
            exclude.add(pos_item)

            negatives = sample_negatives(candidate_pool, exclude, n_negatives=n_negatives, rng=rng)
            candidate_items = np.array([pos_item] + negatives.tolist(), dtype=int)
            positive_mask = np.zeros(len(candidate_items), dtype=bool)
            positive_mask[0] = True

            perm = rng.permutation(len(candidate_items))
            candidate_items = candidate_items[perm]
            positive_mask = positive_mask[perm]

            scores = score_fn(uid, candidate_items)
            rank, tie_count = rank_positive_among_candidates(scores, positive_mask, rng=rng)
            if tie_count > 1:
                tie_cases += 1

            recall, ndcg, mrr = ranking_metrics_from_rank(rank, k=k)
            rows.append((recall, ndcg, mrr))

    if not rows:
        return {"Recall@10": np.nan, "NDCG@10": np.nan, "MRR": np.nan, "n_eval": 0, "tie_rate": np.nan}

    arr = np.array(rows, dtype=float)
    n_eval = int(len(rows))
    return {
        "Recall@10": float(arr[:, 0].mean()),
        "NDCG@10": float(arr[:, 1].mean()),
        "MRR": float(arr[:, 2].mean()),
        "n_eval": n_eval,
        "tie_rate": float(tie_cases / n_eval),
    }

## 5. Build LightFM Dataset and Feature Matrices

In [5]:
pip install lightfm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 7.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lightfm: filename=lightfm-1.17-cp311-cp311-linux_x86_64.whl size=825357 sha256=863c238848611dec6b597badebb96623aa0ff0347585d4186caf2c8300a1444c
  Stored in directory: /root/.cache/pip/wheels/b9/0d/8a/0729d2e6e3ca2a898ba55201f905da7db3f838a33df5b3fcdd
Successfully built lightfm


In [6]:
from lightfm.data import Dataset

def build_item_feature_dicts(movies_df):
    # Build sparse item features from metadata tokens and normalized year.
    movies_local = movies_df.copy()
    year_min = movies_local["year_published"].min(skipna=True)
    year_max = movies_local["year_published"].max(skipna=True)
    year_span = (year_max - year_min) if pd.notna(year_min) and pd.notna(year_max) and year_max != year_min else 1.0
    movies_local["year_norm"] = ((movies_local["year_published"] - year_min) / year_span).fillna(0.0).clip(0.0, 1.0)

    item_feature_dicts = []
    for _, row in movies_local.iterrows():
        genres = split_tokens(row.get("genres", ""))
        actors = split_tokens(row.get("actors", ""))
        directors = split_tokens(row.get("directors", ""))

        feat = {}
        for g in genres:
            feat[f"genre__{g}"] = 1.0
        feat["year_norm"] = float(row["year_norm"])
        for a in actors:
            feat[f"actor__{a}"] = 1.0
        for d in directors:
            feat[f"director__{d}"] = 1.0

        item_feature_dicts.append((int(row["id"]), feat))

    return item_feature_dicts

def build_user_feature_dicts(users_df):
    # Build user features from behavior flags and log-scaled followers.
    users_local = users_df.copy()
    users_local["log_followers"] = np.log1p(pd.to_numeric(users_local["followers_count"], errors="coerce").fillna(0.0))

    user_feature_dicts = []
    for _, row in users_local.iterrows():
        feat = {}
        for flag in flag_cols:
            feat[f"flag__{flag}"] = float(int(row.get(flag, 0)))
        feat["log_followers"] = float(row["log_followers"])
        user_feature_dicts.append((int(row["user_id"]), feat))

    return user_feature_dicts

def build_lightfm_dataset(all_users, all_items, user_feature_dicts, item_feature_dicts):
    # Register all users/items and the feature vocabularies in LightFM.
    dataset = Dataset()
    dataset.fit(
        users=all_users,
        items=all_items,
        user_features=sorted({f for _, d in user_feature_dicts for f in d.keys()}),
        item_features=sorted({f for _, d in item_feature_dicts for f in d.keys()}),
    )
    return dataset

def prepare_interactions(df):
    return [(int(r["user_id"]), int(r["movie_id"])) for _, r in df.iterrows()]

def filter_feature_dicts(feature_dicts, valid_ids):
    valid_id_set = set(valid_ids)
    return [(entity_id, feat) for entity_id, feat in feature_dicts if entity_id in valid_id_set]

all_user_ids = sorted(users["user_id"].dropna().astype(int).unique().tolist())
all_user_id_set = set(all_user_ids)
all_item_id_set = set(all_item_ids)

train_df_lfm = train_df[
    train_df["user_id"].astype(int).isin(all_user_id_set)
    & train_df["movie_id"].astype(int).isin(all_item_id_set)
].copy()

item_features = filter_feature_dicts(build_item_feature_dicts(movies), all_item_id_set)
user_features = filter_feature_dicts(build_user_feature_dicts(users), all_user_id_set)

dataset = build_lightfm_dataset(all_user_ids, all_item_ids, user_features, item_features)
train_interactions, _ = dataset.build_interactions(prepare_interactions(train_df_lfm))
user_features_mx = dataset.build_user_features(user_features)
item_features_mx = dataset.build_item_features(item_features)

user_id_map, _, item_id_map, _ = dataset.mapping()

print(f"LightFM users: {len(user_id_map):,} | items: {len(item_id_map):,}")

LightFM users: 482 | items: 78,628


## 6. Train Vanilla LightFM

In [7]:
from lightfm import LightFM

# WARP is optimized for ranking quality in implicit-feedback setups.
model = LightFM(loss="warp", no_components=64, learning_rate=0.05, random_state=42)
model.fit(
    train_interactions,
    user_features=user_features_mx,
    item_features=item_features_mx,
    epochs=30,
    num_threads=1,
    verbose=True,
)

Epoch: 100%|██████████| 30/30 [11:55<00:00, 23.86s/it]


## 7. Evaluate on Test Split

In [8]:
def vanilla_lfm_score_fn(uid, candidate_items):
    # Score candidate items for one user; unknown IDs receive zero score.
    if uid not in user_id_map:
        return np.zeros(len(candidate_items), dtype=float)

    uidx = user_id_map[uid]
    item_idxs = []
    valid_positions = []

    for pos, item in enumerate(candidate_items):
        if item in item_id_map:
            item_idxs.append(item_id_map[item])
            valid_positions.append(pos)

    scores = np.zeros(len(candidate_items), dtype=float)
    if item_idxs:
        preds = model.predict(
            uidx,
            np.array(item_idxs, dtype=int),
            user_features=user_features_mx,
            item_features=item_features_mx,
        )
        for pos, sc in zip(valid_positions, preds):
            scores[pos] = float(sc)

    return scores

test_metrics = evaluate_ranker(
    vanilla_lfm_score_fn,
    positives_df=test_df,
    candidate_pool=all_item_set,
    all_user_pos_lookup=all_user_pos,
    n_negatives=20,
    k=10,
    seed=43,
)

print("Vanilla LightFM (User Split 80/20)")
print("Test:", test_metrics)

Vanilla LightFM (User Split 80/20)
Test: {'Recall@10': 0.8341552527080457, 'NDCG@10': 0.6436732906905828, 'MRR': 0.5940491036329264, 'n_eval': 233194, 'tie_rate': 0.1550211411957426}
